In [ ]:
# @title Cell 1: Environment Setup & Dependencies
# Single consolidated install cell. Run this once, before any other cell.
# It installs every system package and Python library used by the rest of the notebook.
!sudo apt-get update -qq && sudo apt-get install -y pigz librdkit-dev ripgrep
!pip -q install "rdkit>=2024.9" "numpy>=2.0,<3" pandas pillow matplotlib mols2grid tqdm plotly duckdb pyarrow "ipywidgets>=7,<8" --break-system-packages


In [ ]:
# @title Cell 2: Exit Vector Visualizer (Select your reaction atom)
import os
import ipywidgets as W
from IPython.display import display, Image, HTML, SVG
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
try:
    from rdkit.Chem import rdMolDraw2D as rdmoldraw2d
except ImportError:
    from rdkit.Chem.Draw import rdMolDraw2D as rdmoldraw2d

RDLogger.DisableLog("rdApp.*")

def draw_smiles_with_atom_puff(smiles, width=400, height=400, highlight_atom=None):
    """Draws a 2D molecule with a pastel highlight on a specific atom."""
    m = Chem.MolFromSmiles(smiles)
    if m is None: return None
    Chem.AssignStereochemistry(m, cleanIt=True, force=True, flagPossibleStereoCenters=True)
    AllChem.Compute2DCoords(m)

    drawer = rdmoldraw2d.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addAtomIndices = True
    opts.addStereoAnnotation = True
    opts.atomHighlightsAreCircles = True
    opts.fillHighlights = True

    highlightAtoms, highlightColors, highlightRadii = [], {}, {}
    if highlight_atom is not None:
        idx = int(highlight_atom)
        if m.GetNumAtoms() > idx:
            highlightAtoms = [idx]
            highlightColors[idx] = (1.0, 0.45, 0.50, 0.45) # Pastel Red
            highlightRadii[idx] = 0.4

    rdmoldraw2d.PrepareAndDrawMolecule(drawer, m, highlightAtoms=highlightAtoms,
                                       highlightAtomColors=highlightColors,
                                       highlightAtomRadii=highlightRadii)
    drawer.FinishDrawing()
    return drawer.GetDrawingText()

# --- UI Setup ---
smiles_w = W.Text(value="OC[C@@H]1CNC[C@H]1c2cccc(F)c2", description="SMILES:", layout=W.Layout(width="600px"))
atom_dd  = W.Dropdown(options=[], description="Atom:", layout=W.Layout(width="200px"))
gen_btn  = W.Button(description="Visualize Core", button_style='info')
img_out  = W.Output()

def update_dropdown(smi):
    m = Chem.MolFromSmiles(smi)
    if m:
        opts = [(f"{i}: {a.GetSymbol()}", i) for i, a in enumerate(m.GetAtoms())]
        atom_dd.options = opts
        # Auto-select the first Nitrogen if present
        n_idx = next((i for i, a in enumerate(m.GetAtoms()) if a.GetSymbol() == "N"), opts[0][1] if opts else None)
        atom_dd.value = n_idx

def on_render(_):
    with img_out:
        img_out.clear_output()
        svg_text = draw_smiles_with_atom_puff(smiles_w.value.strip(), highlight_atom=atom_dd.value)
        if svg_text: display(SVG(svg_text))
        else: print("Invalid SMILES string.")

smiles_w.observe(lambda change: update_dropdown(change.new) if change.name == 'value' else None)
gen_btn.on_click(on_render)

# Initialize
update_dropdown(smiles_w.value)
display(W.VBox([W.HBox([smiles_w, atom_dd, gen_btn]), img_out]))
on_render(None)

In [ ]:
#@title Cell 3: Use mols2grid to visulaise a 100 molecule sample of the SMILES in /unclassified_reagents.smi
import os
import pandas as pd
import mols2grid
from IPython.display import display

unclass_path = os.path.join(os.getcwd(), 'unclassified_reagents.smi')

if os.path.exists(unclass_path):
    # Read the first 100 lines
    df_unclass = pd.read_csv(unclass_path, header=None, names=['SMILES'], nrows=100)

    print(f"🖼️ Rendering preview of {len(df_unclass)} unclassified molecules...")
    grid = mols2grid.display(
        df_unclass,
        smiles_col='SMILES',
        n_cols=5,
        use_svg=True,
        tooltip=['SMILES']
    )
    display(grid)
else:
    print(f"⚠️ The file {unclass_path} was not found.")

"""The "next cell" will be the Setup, Configuration, and Reaction Definitions block. This is where you define your target scaffold (Core SMILES), point the enumerator to the freshly cleaned data, and load all the medicinal chemistry reaction rules natively into memory."""

In [ ]:
# @title Cell 4: Global Configuration & Native Reaction Definitions
from rdkit import Chem
from rdkit.Chem import AllChem
import collections

# --- 1. GLOBAL CONFIGURATION ---
# Your chemical scaffold with the target secondary amine [N;R;H1]
CORE_SMILES   = "O=C(O)[C@@H]1CNC[C@H]1c2cccc(F)c2"

# NOTE: Standard Colab instances (~12GB RAM) cannot handle the full multi-billion
# molecule dataset. Point this to a smaller sample file instead.
# You must upload your sample .smi.gz file to /content before running this cell.
BB_FILE       = "/content/sample_zinc_chunk.smi.gz"

# Property Filters (Lipinski-ish)
ENABLE_FILTERS = True
MW_RANGE   = (150.0, 550.0)
LOGP_RANGE = (-1.0, 4.5)
HBD_RANGE  = (0, 3)
HBA_RANGE  = (0, 10)
TPSA_RANGE = (0.0, 120.0)
RB_RANGE   = (0, 10)
FSP3_RANGE = (0.0, 0.8)

print(f"✅ Configuration set. Core SMILES: {CORE_SMILES}")
print(f"✅ Building Blocks target: {BB_FILE}")

# --- 2. SYNCHRONIZED REACTION DEFINITIONS ---
# Mapped strictly to the Top 50 Secondary Amine Reagents.
# Reactant 1 is always the Core Secondary Amine. Reactant 2 is the Building Block.
reaction_data = [
    # --- Acylation & Related Derivatizations ---
    { "name": "Carboxylic Acid (Amide Coupling)", "smarts": "[N;R;H1:1].[CX3:2](=O)[OH]>>[N:1][C:2]=O" },
    { "name": "Sulfonyl Chloride (Sulfonamide Synthesis)", "smarts": "[N;R;H1:1].[Cl][S:2](=O)(=O)>>[N:1][S:2](=O)(=O)" },
    { "name": "Isocyanate (Urea Synthesis)", "smarts": "[N;R;H1:1].[N:3]=[C:2]=[O:4]>>[N:1][C:2](=[O:4])[NH:3]" },
    { "name": "Chloroformate (Carbamate Synthesis)", "smarts": "[N;R;H1:1].[Cl][C:2](=O)[OX2:3]>>[N:1][C:2](=O)[O:3]" },
    { "name": "Dialkyl Dicarbonate (Carbamate Synth)", "smarts": "[N;R;H1:1].[OX2:3][C:2](=O)[OX2]C(=O)[OX2]>>[N:1][C:2](=O)[O:3]" },
    { "name": "Isothiocyanate (Thiourea Synthesis)", "smarts": "[N;R;H1:1].[N:3]=[C:2]=[S:4]>>[N:1][C:2](=[S:4])[NH:3]" },
    { "name": "Acid Chloride (N-Acylation)", "smarts": "[N;R;H1:1].[Cl][C:2](=O)[#6:3]>>[N:1][C:2](=O)[#6:3]" },
    { "name": "Acid Anhydride (N-Acylation)", "smarts": "[N;R;H1:1].[#6:3][C:2](=O)[OX2][CX3](=O)>>[N:1][C:2](=O)[#6:3]" },

    # --- Alkylation & Arylation ---
    { "name": "Aldehyde (Reductive Amination / MCRs)", "smarts": "[N;R;H1:1].[CH1:2](=O)[#6:3]>>[N:1][CH2:2][#6:3]" },
    { "name": "Ketone (Reductive Amination)", "smarts": "[N;R;H1:1].[#6:3][C:2](=O)[#6:4]>>[N:1][CH1:2]([#6:3])[#6:4]" },
    { "name": "Aryl Halide (SNAr / Buchwald-Hartwig)", "smarts": "[N;R;H1:1].[F,Cl,Br,I][c:2]>>[N:1][c:2]" },
    { "name": "Alkyl Halide (Direct N-Alkylation)", "smarts": "[N;R;H1:1].[F,Cl,Br,I][CX4:2]>>[N:1][C:2]" },
    { "name": "Alkyl Sulfonate (Direct N-Alkylation)", "smarts": "[N;R;H1:1].[CX4:2][OX2][S](=O)(=O)>>[N:1][C:2]" },
    { "name": "Boronic Acid / Ester (Chan-Lam / Petasis)", "smarts": "[N;R;H1:1].[B]([O,OH])([O,OH])[#6:2]>>[N:1][#6:2]" },
    { "name": "Carbon Monoxide (Aminocarbonylation)", "smarts": "[N;R;H1:1].[C-:2]#[O+]>>[N:1][CH1:2]=O" },

    # --- Additions & Ring Openings ---
    { "name": "Alpha,beta-Unsaturated Carbonyl", "smarts": "[N;R;H1:1].[O:4]=[C:3]-[C:2]=[C:5]>>[N:1][C:5]-[CH1:2]-[C:3]=[O:4]" },
    { "name": "Epoxide (Ring Opening)", "smarts": "[N;R;H1:1].[O:2]1[C:3][C:4]1>>[N:1][C:3]-[C:4]-[OH:2]" },
    { "name": "Aziridine (Ring Opening)", "smarts": "[N;R;H1:1].[N:2]1[C:3][C:4]1>>[N:1][C:3]-[C:4]-[NH2:2]" },

    # --- Specialized & Multicomponent Reactions (MCRs) ---
    { "name": "Cyanide Ion (Strecker Synthesis)", "smarts": "[N;R;H1:1].[C-:2]#N>>[N:1][C:2]#N" },
    { "name": "Guanylating Agent", "smarts": "[N;R;H1:1].[NX3:3][C:2](=[NX2:4])n1cccc1>>[N:1][C:2](=[N:4])[N:3]" },

    # --- Prodrugs, Linkers & Bioisosteres ---
    { "name": "Sulfamoyl Chloride (Sulfamide Synthesis)", "smarts": "[N;R;H1:1].[Cl][S:2](=O)(=O)[N:3]>>[N:1][S:2](=O)(=O)[N:3]" },
    { "name": "Phosphorochloridate (Phosphoramidate)", "smarts": "[N;R;H1:1].[Cl][P:2](=O)([OX2:3])[OX2:4]>>[N:1][P:2](=O)([O:3])[O:4]" },
    { "name": "Phosgene / Triphosgene", "smarts": "[N;R;H1:1].[Cl][C:2](=O)[Cl:3]>>[N:1][C:2](=O)[Cl:3]" },

    # --- Targeted Covalent Inhibitor (TCI) Warheads ---
    { "name": "Cyanogen Halide (Cyanamide Formation)", "smarts": "[N;R;H1:1].[F,Cl,Br,I][C:2]#[N:3]>>[N:1][C:2]#[N:3]" },
    { "name": "Vinyl Sulfone (Conjugate Addition)", "smarts": "[N;R;H1:1].[C:3]=[C:2]-[S:4](=O)(=O)>>[N:1][C:3]-[CH1:2]-[S:4](=O)(=O)" },

    # --- SuFEx Click Chemistry ---
    { "name": "Sulfonyl Fluoride (SuFEx)", "smarts": "[N;R;H1:1].[F][S:2](=O)(=O)[#6:3]>>[N:1][S:2](=O)(=O)[#6:3]" },
    { "name": "Aryl Fluorosulfate (SuFEx)", "smarts": "[N;R;H1:1].[F][S:2](=O)(=O)[OX2:3][c:4]>>[N:1][S:2](=O)(=O)[O:3][c:4]" },

    # --- Complex Alkylation & MCRs ---
    { "name": "Alpha-Halo Ketone (Alpha-Amination)", "smarts": "[N;R;H1:1].[F,Cl,Br,I][CX4:2][C:3](=O)>>[N:1][C:2][C:3]=O" },
    { "name": "Terminal Alkyne (A³ Coupling)", "smarts": "[N;R;H1:1].[C;H1:3]#[C:2]>>[N:1][C:2]#[C:3]" },
    { "name": "Alkyl Nitrite (N-Nitrosation)", "smarts": "[N;R;H1:1].[OX2][N:2]=[O:3]>>[N:1][N:2]=[O:3]" },

    # --- Advanced Bioisosteres & Heteroatom Chemistry ---
    { "name": "Dialkyl Squarate (Squaramide Bioisostere)", "smarts": "[N;R;H1:1].[OX2]C1=C([OX2:2])C(=O)C1=O>>[N:1]C1=C([O:2])C(=O)C1=O" },
    { "name": "Carbon Disulfide (Dithiocarbamate)", "smarts": "[N;R;H1:1].S=[C:2]=S>>[N:1][C:2](=S)S" },
    { "name": "Imidate (Amidine Synthesis)", "smarts": "[N;R;H1:1].[C:2](=[NX2:3])[OX2]>>[N:1][C:2]=[N:3]" },
    { "name": "Thioacyl Chloride (Thioamide Synthesis)", "smarts": "[N;R;H1:1].[Cl][C:2](=S)[#6:3]>>[N:1][C:2](=S)[#6:3]" },
    { "name": "Oxalyl Chloride (Oxalamide Synthesis)", "smarts": "[N;R;H1:1].[Cl][C:2](=O)[C:3](=O)[Cl]>>[N:1][C:2](=O)[C:3](=O)Cl" },

    # Heteroatom-Nitrogen Bond Formations
    { "name": "Hydroxylamine-O-sulfonic acid (HOSA)", "smarts": "[N;R;H1:1].[NX3:2][OX2][S](=O)(=O)[OH]>>[N:1][N:2]" },
    { "name": "Peroxy Acid (N-Oxidation)", "smarts": "[N;R;H1:1].C(=O)[OX2][OH:2]>>[N:1][OH:2]" },
    { "name": "N-Chlorosuccinimide (N-Chlorination)", "smarts": "[N;R;H1:1].[Cl:2][N]1C(=O)CCC1=O>>[N:1][Cl:2]" },
    { "name": "Sulfinyl Chloride (Sulfinamide Synthesis)", "smarts": "[N;R;H1:1].[Cl][S:2](=O)[#6:3]>>[N:1][S:2](=O)[#6:3]" },
    { "name": "Sulfenyl Chloride (Sulfenamide Synthesis)", "smarts": "[N;R;H1:1].[Cl][SX2:2][#6:3]>>[N:1][S:2][#6:3]" },
    { "name": "Silyl Chloride (N-Silylation)", "smarts": "[N;R;H1:1].[Cl][Si:2]>>[N:1][Si:2]" },

    # Modern Catalysis, Radicals & Aryne Chemistry
    { "name": "Redox-Active Ester / NHP Ester", "smarts": "[N;R;H1:1].[C:2](=O)[OX2][N]1C(=O)c2ccccc2C1=O>>[N:1][C:2]=O" },
    { "name": "ortho-Silyl Aryl Triflate (Benzyne)", "smarts": "[N;R;H1:1].[Si][c:2]1[c:3]([OX2][S](=O)(=O)C(F)(F)F)cccc1>>[N:1][c:2][c:3]" },
    { "name": "Unactivated Alkene (Hydroamination)", "smarts": "[N;R;H1:1].[CX3:2]=[CX3:3]>>[N:1][C:2]-[CH1:3]" },
    { "name": "Activated Alkyne (Conjugate Addition)", "smarts": "[N;R;H1:1].[C:3]#[C:2]C(=O)>>[N:1][C:2]=[CH1:3]C(=O)" },
    { "name": "Isocyanide (Ugi-Smiles MCRs)", "smarts": "[N;R;H1:1].[C-:2]#[N+:3]>>[N:1][C:2]=[N:3]" },

    # Specialized Alkylations & Formylations
    { "name": "Alcohol (Mitsunobu Alkylation)", "smarts": "[N;R;H1:1].[CX4:2][OH]>>[N:1][C:2]" },
    { "name": "Formate Ester (N-Formylation)", "smarts": "[N;R;H1:1].[CH1:2](=O)[OX2]>>[N:1][CH1:2]=O" },
    { "name": "Formic Acid (Eschweiler-Clarke)", "smarts": "[N;R;H1:1].[CH1:2](=O)[OH]>>[N:1][CH3:2]" },
    { "name": "Trifluoromethyl Iodide", "smarts": "[N;R;H1:1].I[C:2](F)(F)F>>[N:1][C:2](F)(F)F" }
]

RXN_DEFINITIONS = collections.defaultdict(list)

print("🧪 Compiling Synchronized Reaction SMARTS...")
for entry in reaction_data:
    name, rtype, smarts = entry["name"], "Amine Derivatization", entry["smarts"]
    try:
        rxn = AllChem.ReactionFromSmarts(smarts)
        if rxn:
            rxn.Initialize()
            RXN_DEFINITIONS[rtype].append((name, rxn))
    except Exception as e:
        print(f"⚠️ Skipping invalid SMARTS for {name}: {e}")

print(f"✅ Successfully loaded {sum(len(v) for v in RXN_DEFINITIONS.values())} reactions mapped to your Reagents List.")

In [ ]:
# @title Cell 5: Dual-Core Scanner & Binner
import os, glob, subprocess, shutil, collections, time, math
import concurrent.futures
import multiprocessing
import queue
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import FilterCatalog
from tqdm.auto import tqdm
from IPython.display import display, FileLink

RDLogger.DisableLog("rdApp.*")

# --- 1. Bulletproofed SMARTS Dictionary ---
functional_groups_smarts = {
    "Carboxylic Acid (Amide Coupling)": "[CX3](=O)[OH]",
    "Sulfonyl Chloride (Sulfonamide Synthesis)": "[Cl][S](=O)(=O)",
    "Isocyanate (Urea Synthesis)": "[N]=[C]=[O]",
    "Chloroformate (Carbamate Synthesis)": "[Cl]C(=O)[OX2]",
    "Dialkyl Dicarbonate (Carbamate Synth)": "[OX2]C(=O)[OX2]C(=O)[OX2]",
    "Isothiocyanate (Thiourea Synthesis)": "[N]=[C]=[S]",
    "Acid Chloride (N-Acylation)": "[Cl]C(=O)[#6]",
    "Acid Anhydride (N-Acylation)": "[CX3](=O)[OX2][CX3](=O)",
    "Aldehyde (Reductive Amination / MCRs)": "[CH1](=O)[#6]",
    "Ketone (Reductive Amination)": "[#6][CX3](=O)[#6]",
    "Aryl Halide (SNAr / Buchwald-Hartwig)": "[F,Cl,Br,I]c",
    "Alkyl Halide (Direct N-Alkylation)": "[F,Cl,Br,I][CX4]",
    "Alkyl Sulfonate (Direct N-Alkylation)": "[CX4][OX2][S](=O)(=O)",
    "Boronic Acid / Ester (Chan-Lam / Petasis)": "[B]([O,OH])([O,OH])",
    "Carbon Monoxide (Aminocarbonylation)": "[C-]#[O+]",
    "Alpha,beta-Unsaturated Carbonyl": "O=[C]-[C]=[C]",
    "Epoxide (Ring Opening)": "[O]1[C][C]1",
    "Aziridine (Ring Opening)": "[N]1[C][C]1",
    "Cyanide Ion (Strecker Synthesis)": "[C-]#N",
    "Guanylating Agent": "[NX3]C(=[NX2])n1cccc1",
    "Sulfamoyl Chloride (Sulfamide Synthesis)": "[Cl][S](=O)(=O)[N]",
    "Phosphorochloridate (Phosphoramidate)": "[Cl][P](=O)([OX2])[OX2]",
    "Phosgene / Triphosgene": "[Cl]C(=O)[Cl]",
    "Cyanogen Halide (Cyanamide Formation)": "[F,Cl,Br,I]C#N",
    "Vinyl Sulfone (Conjugate Addition)": "[C]=[C]-[S](=O)(=O)",
    "Sulfonyl Fluoride (SuFEx)": "[F][S](=O)(=O)",
    "Aryl Fluorosulfate (SuFEx)": "[F][S](=O)(=O)[OX2]c",
    "Alpha-Halo Ketone (Alpha-Amination)": "[F,Cl,Br,I][CX4]C(=O)",
    "Terminal Alkyne (A³ Coupling)": "[C;H1]#[C]",
    "Alkyl Nitrite (N-Nitrosation)": "[OX2][N]=[O]",
    "Dialkyl Squarate (Squaramide Bioisostere)": "[OX2]C1=C([OX2])C(=O)C1=O",
    "Carbon Disulfide (Dithiocarbamate)": "S=C=S",
    "Imidate (Amidine Synthesis)": "C(=[NX2])[OX2]",
    "Thioacyl Chloride (Thioamide Synthesis)": "[Cl]C(=S)[#6]",
    "Oxalyl Chloride (Oxalamide Synthesis)": "[Cl]C(=O)C(=O)[Cl]",
    "Hydroxylamine-O-sulfonic acid (HOSA)": "[NX3][OX2][S](=O)(=O)[OH]",
    "Peroxy Acid (N-Oxidation)": "C(=O)[OX2][OH]",
    "N-Chlorosuccinimide (N-Chlorination)": "Cl[N]1C(=O)CCC1=O",
    "Sulfinyl Chloride (Sulfinamide Synthesis)": "[Cl][S](=O)[#6]",
    "Sulfenyl Chloride (Sulfenamide Synthesis)": "[Cl][SX2][#6]",
    "Silyl Chloride (N-Silylation)": "[Cl][Si]",
    "Redox-Active Ester / NHP Ester": "C(=O)[OX2][N]1C(=O)c2ccccc2C1=O",
    "ortho-Silyl Aryl Triflate (Benzyne)": "[Si]c1c([OX2][S](=O)(=O)C(F)(F)F)cccc1",
    "Unactivated Alkene (Hydroamination)": "[CX3]=[CX3]",
    "Activated Alkyne (Conjugate Addition)": "[C]#[C]C(=O)",
    "Isocyanide (Ugi-Smiles MCRs)": "[C-]#[N+]",
    "Alcohol (Mitsunobu Alkylation)": "[CX4][OH]",
    "Formate Ester (N-Formylation)": "[CH1](=O)[OX2]",
    "Formic Acid (Eschweiler-Clarke)": "[CH1](=O)[OH]",
    "Trifluoromethyl Iodide": "I[C](F)(F)F"
}

# --- 2. Path Setup ---
base_dir = os.getcwd()
cleaned_file_path = os.path.join(base_dir, 'zinc22_raw', 'combined_cleaned.smi.gz')
ram_disk_raw = "/content/raw_mols.smi"
bin_dir = os.path.join(base_dir, "binned_bbs")

os.makedirs(bin_dir, exist_ok=True)
print("🗑️ Clearing old output files...")
for old_file in glob.glob(os.path.join(bin_dir, "*.smi")):
    try: os.remove(old_file)
    except: pass
for old_part in glob.glob("/content/mol_chunk_*"):
    try: os.remove(old_part)
    except: pass

# --- 3. The Isolated Worker (Verbose & Chunked) ---
def hyper_scale_worker(worker_id, chunk_file, smarts_dict, prog_queue):
    local_catalog = FilterCatalog.FilterCatalog()
    local_writers = {}
    total_found = 0
    internal_scanned = 0

    for rname, smarts in smarts_dict.items():
        safe_name = rname.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
        matcher = FilterCatalog.SmartsMatcher(safe_name, smarts, 1)
        local_catalog.AddEntry(FilterCatalog.FilterCatalogEntry(safe_name, matcher))
        local_writers[safe_name] = open(os.path.join(bin_dir, f"{safe_name}_{worker_id}.part"), 'w')

    unclass_file = open(os.path.join(bin_dir, f"unclass_{worker_id}.part"), 'w')

    # Standard Python file iteration — no DuckDB / Arrow needed.
    # Each line is whitespace-separated: SMILES<TAB-or-SPACE>ZINC_ID[ ...]
    PROGRESS_TICK = 5000  # push progress every 5,000 molecules (smooth on 2 cores)
    pending = 0

    try:
        with open(chunk_file, 'r') as f:
            for line in f:
                line = line.rstrip('\n').rstrip('\r')
                if not line:
                    continue

                # Split by tab first (matches the writers in this notebook); fall back
                # to any whitespace for ZINC22 raw files which are space-separated.
                parts = line.split('\t')
                if len(parts) < 2:
                    parts = line.split()
                if len(parts) < 2:
                    continue
                smi, zid = parts[0], parts[1]
                if not smi:
                    continue

                mol = Chem.MolFromSmiles(smi)
                internal_scanned += 1
                pending += 1

                if mol:
                    matches = local_catalog.GetMatches(mol)
                    if matches:
                        total_found += 1
                        for m in matches:
                            local_writers[m.GetDescription()].write(f"{smi}\t{zid}\n")
                    else:
                        unclass_file.write(f"{smi}\t{zid}\n")

                # Push progress every PROGRESS_TICK molecules so the bar stays smooth on 2 cores
                if pending >= PROGRESS_TICK:
                    prog_queue.put((pending, total_found))
                    pending = 0

                # VERBOSITY: Print a heartbeat every 250k molecules
                if internal_scanned % 250000 == 0:
                    print(f"🟢 [Worker {worker_id:02d}] Scanned: {internal_scanned:,} | Caught: {total_found:,}")

        # Flush any remaining progress
        if pending > 0:
            prog_queue.put((pending, total_found))

    except Exception as e:
        print(f"❌ Worker {worker_id} Error: {e}")

    for w in local_writers.values(): w.close()
    unclass_file.close()
    return total_found

# --- 4. Execution Pipeline ---
num_workers = os.cpu_count() or 2

# Step 4A: Decompress
print(f"\n🚀 Step 1/3: Decompressing molecules to local /content disk...")
subprocess.run(f"unpigz -p {num_workers} -c '{cleaned_file_path}' > {ram_disk_raw}", shell=True, check=True)
total_mols = int(subprocess.check_output(f"wc -l {ram_disk_raw}", shell=True).split()[0])
lines_per_chunk = math.ceil(total_mols / num_workers)

# Step 4B: Chop the file
print(f"🔪 Chopping master file into {num_workers} pieces for zero-latency reading...")
subprocess.run(f"split -d -l {lines_per_chunk} {ram_disk_raw} /content/mol_chunk_", shell=True, check=True)
os.remove(ram_disk_raw) # Free up disk immediately

chunk_files = sorted(glob.glob("/content/mol_chunk_*"))
print(f"🚀 Step 2/3: Launching {num_workers} Workers (Processing {total_mols:,} molecules)...")
print("-" * 60)

start_time = time.time()
manager = multiprocessing.Manager()
prog_queue = manager.Queue()
total_reagents_caught = 0

with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers) as executor:
    # Launch workers
    futures = [executor.submit(hyper_scale_worker, i, chunk_files[i], functional_groups_smarts, prog_queue) for i in range(len(chunk_files))]

    # Live Progress Bar Thread
    with tqdm(total=total_mols, desc="Scanning & Binning", unit="mols") as pbar:
        while not all(f.done() for f in futures):
            try:
                # Pull ticks from the queue and update the bar smoothly
                scanned_tick, current_caught = prog_queue.get(timeout=2.0)
                pbar.update(scanned_tick)
                pbar.set_postfix({"Reagents Caught": f"{current_caught:,}"})
            except queue.Empty:
                continue

# Add up the final exact totals
final_caught = sum(f.result() for f in futures)

# --- 5. Atomic OS Merge ---
print("\n" + "-" * 60)
print("🚀 Step 3/3: Merging worker fragments into final bins...")
for rname in functional_groups_smarts.keys():
    safe_name = rname.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
    if glob.glob(os.path.join(bin_dir, f"{safe_name}_*.part")):
        subprocess.run(f"cat {bin_dir}/{safe_name}_*.part > {bin_dir}/{safe_name}.smi && rm {bin_dir}/{safe_name}_*.part", shell=True)
subprocess.run(f"cat {bin_dir}/unclass_*.part > {base_dir}/unclassified_reagents.smi && rm {bin_dir}/unclass_*.part", shell=True)

# Cleanup /content chunk files
for chunk in chunk_files:
    if os.path.exists(chunk): os.remove(chunk)

elapsed = time.time() - start_time
print(f"✅ SUCCESS: {total_mols:,} molecules processed in {elapsed:.1f}s (~{total_mols/elapsed:,.0f} mols/sec).")
print(f"💎 Total Reagents Successfully Binned: {final_caught:,}")

In [ ]:
# @title Cell 6: Preview a reaction bin
import os
import pandas as pd
import mols2grid
from IPython.display import display

# Directly specify the target file
target_file = os.path.join(os.getcwd(), "binned_bbs", "Aryl_Fluorosulfate_SuFEx.smi")

if os.path.exists(target_file):
    print(f"🖼️ Rendering preview of: {os.path.basename(target_file)}")

    # Read the first 100 rows
    df = pd.read_csv(target_file, sep='\t', header=None, names=['SMILES', 'ZINC_ID'], nrows=100)

    # Display with mols2grid
    grid = mols2grid.display(
        df,
        smiles_col='SMILES',
        tooltip=['SMILES', 'ZINC_ID'],
        n_cols=5,
        use_svg=True
    )
    display(grid)
else:
    print(f"⚠️ The file {target_file} was not found.")

In [ ]:
#@title Cell 7: Count total reagents
import os
import glob
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, FileLink

# Configure Plotly to render correctly in Colab
pio.renderers.default = 'colab'

bin_dir = os.path.join(os.getcwd(), "binned_bbs")
smi_files = glob.glob(os.path.join(bin_dir, "*.smi"))
print(f"Found {len(smi_files)} .smi files in {bin_dir}")

data = []
for file_path in smi_files:
    filename = os.path.basename(file_path)
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            count = sum(1 for _ in f)
        data.append({"File": filename, "SMILES Count": count})
    except Exception as e:
        print(f"Error reading {filename}: {e}")

if data:
    df = pd.DataFrame(data)
    df = df.sort_values(by="SMILES Count", ascending=False)
    max_val = df["SMILES Count"].max()

    # Generate CSV
    csv_filename = "reaction_bin_counts.csv"
    df.to_csv(csv_filename, index=False)
    print(f"✅ Data saved to {csv_filename}")

    # Display table preview
    display(df)

    # Generate Plot
    fig = px.bar(
        df,
        x="File",
        y="SMILES Count",
        title="Number of SMILES in each binned .smi file (Log Scale)",
        labels={"File": "Reaction Bin", "SMILES Count": "Number of Molecules"},
        color="SMILES Count",
        color_continuous_scale="Viridis",
        log_y=True,
        height=800,
        range_y=[1, max_val * 2]
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

    # Provide Download Link
    display(FileLink(csv_filename))
else:
    print(f"No .smi files found in {bin_dir}.")

In [ ]:
# @title Cell 8: Build the Industrial C++ Engine (Hyper-Optimized)
import os, subprocess, time

# RDKit C++ development headers (librdkit-dev) are installed in Cell 1.

cpp_source = """
#include <iostream>
#include <fstream>
#include <string>
#include <vector>
#include <memory>
#include <boost/shared_ptr.hpp>
#include <GraphMol/RDKitBase.h>
#include <GraphMol/SmilesParse/SmilesParse.h>
#include <GraphMol/SmilesParse/SmilesWrite.h>
#include <GraphMol/ChemReactions/Reaction.h>
#include <GraphMol/ChemReactions/ReactionParser.h>
#include <GraphMol/Descriptors/MolDescriptors.h>
#include <GraphMol/MolOps.h>

using namespace RDKit;

int main(int argc, char* argv[]) {
    if (argc < 10) return 1;

    std::string core_smi = argv[1], rxn_smarts = argv[2], rxn_name = argv[3];
    std::string input_path = argv[4], output_path = argv[5];
    double mw_min = std::stod(argv[6]), mw_max = std::stod(argv[7]), logp_min = std::stod(argv[8]), logp_max = std::stod(argv[9]);

    std::unique_ptr<ChemicalReaction> rxn(RxnSmartsToChemicalReaction(rxn_smarts));
    if (!rxn) return 1;
    rxn->initReactantMatchers();

    boost::shared_ptr<ROMol> core(SmilesToMol(core_smi));
    if (!core) return 1;

    std::ifstream infile(input_path);
    std::ofstream outfile(output_path);

    // OPTIMIZATION 4: Hijack stream buffers for massive 1MB block reads/writes
    char in_buffer[1048576];
    char out_buffer[1048576];
    infile.rdbuf()->pubsetbuf(in_buffer, sizeof(in_buffer));
    outfile.rdbuf()->pubsetbuf(out_buffer, sizeof(out_buffer));

    outfile << "smiles,zid,reactant_smiles,mw,clogp,rxn_name\\n";

    // OPTIMIZATION 2: Pre-allocate the Reactants vector OUTSIDE the loop
    MOL_SPTR_VECT reactants(2);
    reactants[0] = core;

    std::string line;
    while (std::getline(infile, line)) {
        size_t tab_pos = line.find('\\t');
        if (tab_pos == std::string::npos) continue;

        // OPTIMIZATION 3: Zero-Copy Pointer Arithmetic (No std::string allocations)
        line[tab_pos] = '\\0';
        const char* smi_ptr = line.c_str();
        const char* zid_ptr = line.c_str() + tab_pos + 1;

        boost::shared_ptr<ROMol> partner(SmilesToMol(smi_ptr));
        if (!partner) continue;

        // O(1) vector replacement
        reactants[1] = partner;

        std::vector<MOL_SPTR_VECT> prods = rxn->runReactants(reactants);
        for (auto& prod_vect : prods) {
            for (auto& p : prod_vect) {
                try { p->updatePropertyCache(false); } catch (...) { continue; }

                double mw = Descriptors::calcAMW(*p);
                if (mw < mw_min || mw > mw_max) continue;
                double clogp = Descriptors::calcClogP(*p);
                if (clogp < logp_min || clogp > logp_max) continue;

                try {
                    MolOps::sanitizeMol(*static_cast<RWMol*>(p.get()));
                    outfile << MolToSmiles(*p) << "," << zid_ptr << "," << smi_ptr << "," << mw << "," << clogp << "," << rxn_name << "\\n";
                } catch (...) { continue; }
            }
        }
    }
    return 0;
}
"""

with open("high_speed_enumerator.cpp", "w") as f:
    f.write(cpp_source)

print("🛠️ Compiling Bare-Metal Reactant Engine...")

# OPTIMIZATION 1: Added -march=native, -mtune=native, and -funroll-loops
compile_cmd = [
    "g++", "-O3", "-march=native", "-mtune=native", "-funroll-loops",
    "-I/usr/include/rdkit", "high_speed_enumerator.cpp", "-o", "high_speed_enumerator",
    "-L/usr/lib", "-lRDKitDescriptors", "-lRDKitChemReactions", "-lRDKitSmilesParse",
    "-lRDKitGraphMol", "-lRDKitDataStructs", "-lRDKitRDGeometryLib", "-lRDKitRDGeneral"
]

subprocess.run(compile_cmd, check=True)
os.chmod("high_speed_enumerator", 0o755)
print("✅ Industrial C++ binary compiled with hardware-specific SIMD instructions.")

In [ ]:
# @title Cell 9: C++ Orchestrator & DuckDB Merge
import os, glob, subprocess, duckdb, shutil, time
import pandas as pd
import mols2grid
from IPython.display import FileLink, display
from rdkit.Chem import AllChem
from tqdm.auto import tqdm

# --- 1. Path Setup & Local Disk Initialization ---
base_dir = os.getcwd()
bin_dir = os.path.join(base_dir, "binned_bbs")
final_csv = os.path.join(base_dir, "enumerated_products_final.csv")
local_backup_dir = os.path.join(base_dir, "Billion_Scale_Enumeration_Backups")

ram_in = "/content/cpp_inputs"
ram_out = "/content/cpp_outputs"
tasks_file = "/content/tasks.sh"
ticks_file = "/content/ticks.log"

os.makedirs(local_backup_dir, exist_ok=True)
os.makedirs(ram_in, exist_ok=True)
os.makedirs(ram_out, exist_ok=True)

print("🗑️ Clearing old environments and temp /content directories...")
if os.path.exists(final_csv):
    try: os.remove(final_csv)
    except: pass
subprocess.run(f"rm -rf {ram_in}/* {ram_out}/* {tasks_file} {ticks_file}", shell=True)  # /content paths

num_workers = os.cpu_count() or 2
CHUNK_SIZE = 50000

# --- 2. Zero-Copy Routing & Task Generation ---
print("🔪 Routing bins and generating OS-level execution map...")

commands = []
total_tasks = 0

for rtype, rxn_list in RXN_DEFINITIONS.items():
    for rname, rxn in rxn_list:
        safe_name = rname.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
        clean_rname = rname.replace(",", "_") # Fix for CSV column misalignment
        bin_file = os.path.join(bin_dir, f"{safe_name}.smi")

        if not os.path.exists(bin_file): continue

        try:
            line_count = int(subprocess.check_output(f'wc -l "{bin_file}"', shell=True).split()[0])
        except: continue

        if line_count == 0: continue

        chunk_prefix = os.path.join(ram_in, f"{safe_name}_chunk_")
        rxn_smarts = AllChem.ReactionToSmarts(rxn)

        if line_count <= CHUNK_SIZE:
            single_chunk = f"{chunk_prefix}0000"
            os.symlink(bin_file, single_chunk)
            output_file = os.path.join(ram_out, f"{safe_name}_out_0000.csv")

            cmd = f"./high_speed_enumerator '{CORE_SMILES}' '{rxn_smarts}' '{clean_rname}' '{single_chunk}' '{output_file}' {MW_RANGE[0]} {MW_RANGE[1]} {LOGP_RANGE[0]} {LOGP_RANGE[1]}"
            msg = f"🟢 [Core Thread] Completed routing {line_count:,} molecules for {clean_rname}"
            commands.append(f"{cmd} && echo '{msg}' >> {ticks_file}")
            total_tasks += 1

        else:
            subprocess.run(f"split -d -a 4 -l {CHUNK_SIZE} '{bin_file}' '{chunk_prefix}'", shell=True)
            generated_chunks = sorted(glob.glob(f"{chunk_prefix}*"))

            for i, chunk_file in enumerate(generated_chunks):
                output_file = os.path.join(ram_out, f"{safe_name}_out_{i:04d}.csv")
                cmd = f"./high_speed_enumerator '{CORE_SMILES}' '{rxn_smarts}' '{clean_rname}' '{chunk_file}' '{output_file}' {MW_RANGE[0]} {MW_RANGE[1]} {LOGP_RANGE[0]} {LOGP_RANGE[1]}"
                msg = f"🟢 [Core Thread] Processed 50k micro-chunk ({i+1}/{len(generated_chunks)}) for {clean_rname}"
                commands.append(f"{cmd} && echo '{msg}' >> {ticks_file}")
                total_tasks += 1

with open(tasks_file, "w") as f:
    for c in commands:
        f.write(f"{c}\n")

# --- 3. Bare-Metal Execution (Verbose Streaming) ---
print(f"\n🚀 Launching {num_workers}-Core OS-Level Scheduler on {total_tasks:,} micro-chunks...")
subprocess.run(f"touch {ticks_file}", shell=True)

os_process = subprocess.Popen(
    f"xargs -d '\\n' -P {num_workers} -I {{}} bash -c {{}} < {tasks_file}",
    shell=True
)

with open(ticks_file, 'r') as log_stream:
    with tqdm(total=total_tasks, desc="C++ Orchestration", unit="chunks") as pbar:
        while os_process.poll() is None:
            new_lines = log_stream.readlines()
            for line in new_lines:
                tqdm.write(line.strip())
                pbar.update(1)

            if not new_lines:
                time.sleep(0.2)

        for line in log_stream.readlines():
            tqdm.write(line.strip())
            pbar.update(1)

# --- 4. Unified Output Aggregation & DuckDB Overclocking ---
print("\nΔ Fusing outputs and overclocking DuckDB deduplication...")

merged_ram_file = "/content/merged_outputs.csv"
subprocess.run(f"cat {ram_out}/*.csv > {merged_ram_file} 2>/dev/null", shell=True)

con = duckdb.connect()
con.execute(f"PRAGMA threads={num_workers};")
con.execute(f"PRAGMA memory_limit='10GB';")

# FIXED: All columns are now VARCHAR to bypass parse-crashing on the embedded headers
schema = "{'smiles': 'VARCHAR', 'zid': 'VARCHAR', 'reactant_smiles': 'VARCHAR', 'mw': 'VARCHAR', 'clogp': 'VARCHAR', 'rxn_name': 'VARCHAR'}"
duck_query = f"""
    COPY (
        SELECT smiles, ANY_VALUE(zid) as zid, ANY_VALUE(reactant_smiles) as reactant_smiles,
               ANY_VALUE(mw) as mw, ANY_VALUE(clogp) as clogp, ANY_VALUE(rxn_name) as rxn_name
        FROM read_csv('{merged_ram_file}', columns={schema}, header=true, ignore_errors=true)
        WHERE smiles != 'smiles'
        GROUP BY smiles
    ) TO '{final_csv}' (HEADER, DELIMITER ',');
"""

try:
    if os.path.exists(merged_ram_file) and os.path.getsize(merged_ram_file) > 0:
        con.execute(duck_query)

        backup_path = os.path.join(local_backup_dir, "enumerated_products_final.csv")
        shutil.copy(final_csv, backup_path)

        total_lines = int(subprocess.check_output(f'wc -l "{final_csv}"', shell=True).split()[0])
        total_smiles = total_lines - 1 if total_lines > 0 else 0

        print(f"✅ Success! Generated {total_smiles:,} unique, filtered molecules.")
        display(FileLink(final_csv))

        # --- 5. Visual Dashboard ---
        df_final = pd.read_csv(final_csv, nrows=100)
        if 'smiles' in df_final.columns:
            print(f"\n🖼️ Rendering preview of the first {len(df_final)} generated products...")
            grid = mols2grid.display(
                df_final, smiles_col='smiles',
                subset=['img', 'zid', 'rxn_name', 'mw', 'clogp'],
                tooltip=['smiles', 'reactant_smiles', 'zid', 'rxn_name', 'mw', 'clogp'],
                n_cols=5, use_svg=True
            )
            display(grid)
    else:
        print("⚠️ No valid products were generated. File was empty.")
except Exception as e:
    print(f"⚠️ Merge failed: {e}")
finally:
    subprocess.run(f"rm -rf {ram_in}/* {ram_out}/* {tasks_file} {ticks_file} {merged_ram_file}", shell=True)  # /content paths

In [ ]:
# @title Cell 10: Gold Standard Lead-Like Filter (V16: Unbreakable & Smooth ETA)
import os, glob, subprocess, duckdb, shutil, time, math
import pandas as pd
import mols2grid
from IPython.display import FileLink, display
from tqdm.auto import tqdm

# ==============================================================================
# 🎛️ MEDICINAL CHEMISTRY THRESHOLDS
# ==============================================================================
MAX_ROTATABLE_BONDS = 10
MAX_TPSA            = 140.0
MIN_FSP3            = 0.30
MAX_HBD             = 5
MAX_HBA             = 10
# PAINS, BRENK, and NIH structural toxicity alerts are permanently ON.
# ==============================================================================

input_csv = os.path.join(os.getcwd(), "Billion_Scale_Enumeration_Backups", "enumerated_products_final.csv")
filtered_csv = os.path.join(os.getcwd(), "lead_like_products.csv")

print(f"⚙️ Target: {input_csv}")
print(f"⚙️ Compiling V16 C++ Engine (Micro-Pulse Telemetry)...")

cpp_source = """
#include <iostream>
#include <fstream>
#include <string>
#include <memory>
#include <GraphMol/RDKitBase.h>
#include <GraphMol/SmilesParse/SmilesParse.h>
#include <GraphMol/Descriptors/MolDescriptors.h>
#include <GraphMol/FilterCatalog/FilterCatalog.h>

using namespace RDKit;

int main(int argc, char* argv[]) {
    if (argc < 9) return 1;
    std::string input_path = argv[1];
    std::string output_path = argv[2];

    int max_rot_bonds = std::stoi(argv[3]);
    double max_tpsa = std::stod(argv[4]);
    double min_fsp3 = std::stod(argv[5]);
    int max_hbd = std::stoi(argv[6]);
    int max_hba = std::stoi(argv[7]);
    std::string ticks_file = argv[8];

    FilterCatalogParams params;
    params.addCatalog(FilterCatalogParams::PAINS);
    params.addCatalog(FilterCatalogParams::BRENK);
    params.addCatalog(FilterCatalogParams::NIH);
    FilterCatalog toxicity_catalog(params);

    std::ifstream infile(input_path);
    std::ofstream outfile(output_path);

    std::string line;
    int mol_count = 0;

    while (std::getline(infile, line)) {
        // Gracefully skip the header if it lands in this chunk
        if (line.find("smiles") == 0) continue;

        mol_count++;

        // --- MICRO-PULSE TELEMETRY ---
        // Every 10,000 molecules, append a '1' to the log file natively
        if (mol_count % 10000 == 0) {
            std::ofstream log(ticks_file, std::ios_base::app);
            log << "1\\n";
        }

        size_t comma_pos = line.find(',');
        if (comma_pos == std::string::npos) continue;

        line[comma_pos] = '\\0';
        const char* smi_ptr = line.c_str();

        std::unique_ptr<ROMol> mol(SmilesToMol(smi_ptr));
        if (!mol) continue;

        // --- OPTIMIZED GAUNTLET ---
        if (Descriptors::calcNumRotatableBonds(*mol) > max_rot_bonds) continue;
        if (Descriptors::calcTPSA(*mol) > max_tpsa) continue;
        if (Descriptors::calcFractionCSP3(*mol) < min_fsp3) continue;
        if (Descriptors::calcNumHBD(*mol) > max_hbd) continue;
        if (Descriptors::calcNumHBA(*mol) > max_hba) continue;
        if (toxicity_catalog.hasMatch(*mol)) continue;

        line[comma_pos] = ',';
        outfile << line << "\\n";
    }
    return 0;
}
"""

with open("molecule_filter.cpp", "w") as f:
    f.write(cpp_source)

compile_cmd = [
    "g++", "-O3", "-march=native", "-mtune=native", "-funroll-loops",
    "-I/usr/include/rdkit", "molecule_filter.cpp", "-o", "molecule_filter",
    "-L/usr/lib", "-lRDKitDescriptors", "-lRDKitFilterCatalog", "-lRDKitCatalogs", "-lRDKitSmilesParse",
    "-lRDKitGraphMol", "-lRDKitDataStructs", "-lRDKitRDGeometryLib", "-lRDKitRDGeneral"
]
subprocess.run(compile_cmd, check=True)
os.chmod("molecule_filter", 0o755)
print("✅ C++ Filter Engine compiled successfully.")

# --- 3. NVMe Block-Seeking & 1-to-1 Core Mapping ---
ram_in = "/content/filter_in"
ram_out = "/content/filter_out"
ticks_file = "/content/filter_ticks.log"

subprocess.run(f"rm -rf {ram_in} {ram_out} {ticks_file}", shell=True)
os.makedirs(ram_in, exist_ok=True)
os.makedirs(ram_out, exist_ok=True)

num_workers = os.cpu_count() or 2

print(f"\n📊 Scanning total molecule count for telemetry...")
try:
    total_lines = int(subprocess.check_output(f'wc -l "{input_csv}"', shell=True).split()[0]) - 1
except:
    total_lines = 67204042

print(f"🔪 Phase 1: Zero-Latency NVMe Splitting (Instant {num_workers}-way split)...")
# INSTANT SPLIT: Bypasses Python entirely to slice the file natively on the SSD
subprocess.run(f"split -d -a 2 -n l/{num_workers} '{input_csv}' {ram_in}/chunk_", shell=True)

chunks = sorted(glob.glob(f"{ram_in}/chunk_*"))
commands = []
for i, chunk_file in enumerate(chunks):
    out_file = f"{ram_out}/out_{i:02d}.csv"
    cmd = f"./molecule_filter '{chunk_file}' '{out_file}' {MAX_ROTATABLE_BONDS} {MAX_TPSA} {MIN_FSP3} {MAX_HBD} {MAX_HBA} '{ticks_file}' 2>/dev/null"
    commands.append(cmd)

# --- 5. Bare-Metal Execution (Unbreakable Progress Bar) ---
print(f"\n🚀 Phase 2: Launching {num_workers}-Core Marathon...")
subprocess.run(f"touch {ticks_file}", shell=True)

active_processes = [subprocess.Popen(cmd, shell=True) for cmd in commands]

last_milestone = 0
with tqdm(total=total_lines, desc="Filtering", unit="mols", smoothing=0.1) as pbar:
    # Loop runs as long as ANY core is still alive
    while any(p.poll() is None for p in active_processes):
        time.sleep(0.5) # Poll every half second

        try:
            # OS-LEVEL POLLING: Bypasses Python's EOF trap by asking Linux directly
            ticks = int(subprocess.check_output(f"wc -l < {ticks_file}", shell=True))
            current_mols = ticks * 10000

            # Smoothly catch the bar up to reality
            if current_mols > pbar.n:
                pbar.update(current_mols - pbar.n)

            # VERBOSE FEEDBACK: Print an update above the bar every 5 Million molecules
            if current_mols - last_milestone >= 5000000:
                tqdm.write(f"⚡ [Status] {current_mols:,} molecules scanned... Cores are blazing!")
                last_milestone = current_mols

        except Exception:
            pass

    # Final sweep
    try:
        ticks = int(subprocess.check_output(f"wc -l < {ticks_file}", shell=True))
        current_mols = ticks * 10000
        if current_mols > pbar.n:
            pbar.update(current_mols - pbar.n)
    except: pass

    # Force to 100% to account for small remainders under 10k
    if pbar.n < total_lines:
        pbar.update(total_lines - pbar.n)

# --- 6. Unified RAM-Block Aggregation & DuckDB Overclocking ---
print("\nΔ Re-merging surviving molecules into final Lead-Like CSV...")
merged_ram_file = "/content/merged_filtered.csv"
subprocess.run(f"cat {ram_out}/*.csv > {merged_ram_file} 2>/dev/null", shell=True)

header = subprocess.check_output(f"head -n 1 '{input_csv}'", shell=True).decode('utf-8').strip()
subprocess.run(f"echo '{header}' > '{filtered_csv}'", shell=True)
subprocess.run(f"cat '{merged_ram_file}' >> '{filtered_csv}'", shell=True)

original_lines = total_lines
try:
    surviving_lines = int(subprocess.check_output(f'wc -l "{filtered_csv}"', shell=True).split()[0]) - 1
except:
    surviving_lines = 0
dropped = original_lines - surviving_lines

print(f"\n✅ FILTER COMPLETE!")
print(f"📈 Starting Molecules: {original_lines:,}")
print(f"🗑️ Dropped (Failed Gold Standard Limits): {dropped:,}")
print(f"💎 Surviving Lead-Like Molecules: {surviving_lines:,}")

display(FileLink(filtered_csv))

# --- 7. Visual Dashboard ---
df_final = pd.read_csv(filtered_csv, nrows=100)
if not df_final.empty:
    print(f"\n🖼️ Rendering preview of the surviving Lead-Like products...")
    grid = mols2grid.display(
        df_final, smiles_col='smiles',
        subset=['img', 'zid', 'rxn_name', 'mw', 'clogp'],
        tooltip=['smiles', 'reactant_smiles', 'zid', 'rxn_name', 'mw', 'clogp'],
        n_cols=5, use_svg=True
    )
    display(grid)

subprocess.run(f"rm -rf {ram_in} {ram_out} {ticks_file} {merged_ram_file} molecule_filter*", shell=True)

In [ ]:
#@title Cell 11: Compress and Download lead_like_products.csv
import os
import subprocess
from IPython.display import FileLink, display

csv_path = os.path.join(os.getcwd(), 'lead_like_products.csv')
gz_path = csv_path + '.gz'

if os.path.exists(csv_path):
    print("🗜️ Compressing the Lead-Like products CSV (this might take a minute)...")
    # Using pigz for fast multi-core compression, keeping the original file (-k)
    subprocess.run(f"pigz -k -f -p 2 '{csv_path}' || gzip -k -f '{csv_path}'", shell=True)

    if os.path.exists(gz_path):
        size_mb = os.path.getsize(gz_path) / (1024 * 1024)
        print(f"✅ Compression complete! GZIP File size: {size_mb:.2f} MB")
        print("⬇️ Click the link below to download:")
        display(FileLink(gz_path))
    else:
        print("⚠️ Compression failed.")
else:
    print(f"⚠️ File not found: {csv_path}")